In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CARREGAMENTO E FILTRAGEM
FILE_PATH = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
df = pd.read_csv(FILE_PATH, low_memory=False)

# Identificação dinâmica das colunas
col_id = 'Record ID'
col_ciap = next((c for c in df.columns if 'ciap' in c.lower() or 'diagnostico' in c.lower()), None)
col_status = next((c for c in df.columns if any(p in c.lower() for p in ['desfecho', 'status'])), None)

# Base: Apenas quem foi atendido (exclui faltas)
df_atend = df[~df[col_status].astype(str).str.contains('não compareceu|falta', case=False, na=False)].copy()

# 2. DEFINIÇÃO DOS SUPERUSUÁRIOS (LIMIAR P95)
# Contagem de consultas por paciente
perfil = df_atend.groupby(col_id).size().reset_index(name='Total_Consultas')
limiar_p95 = perfil['Total_Consultas'].quantile(0.95)

# Filtro do grupo de elite de usuários
super_ids = perfil[perfil['Total_Consultas'] >= limiar_p95][col_id]
df_super = df_atend[df_atend[col_id].isin(super_ids)].copy()

# 3. MAPEAMENTO CLÍNICO (CIAP-2)
ciap_nomes = {
    'A': 'Geral/Inespecífico', 'D': 'Digestivo', 'K': 'Circulatório',
    'L': 'Musculoesquelético', 'P': 'Psicológico', 'R': 'Respiratório',
    'S': 'Pele', 'T': 'Endócrino/Metabólico', 'B': 'Sangue'
}
df_super['Capitulo_CIAP'] = df_super[col_ciap].apply(lambda x: ciap_nomes.get(str(x)[0].upper(), 'Outros/Não Informado'))

# ==============================================================================
# VISUALIZAÇÕES
# ==============================================================================
plt.figure(figsize=(15, 6))

# Gráfico 1: Histograma de Frequência de Uso
plt.subplot(1, 2, 1)
sns.histplot(perfil['Total_Consultas'], bins=20, kde=True, color='#2c3e50')
plt.axvline(limiar_p95, color='#e74c3c', linestyle='--', label=f'Limiar P95 (>= {limiar_p95:.0f} consultas)')
plt.title("Distribuição da Intensidade de Uso do Serviço", fontweight='bold')
plt.xlabel("Consultas por Paciente")
plt.ylabel("Nº de Pacientes")
plt.legend()

# Gráfico 2: Perfil Clínico dos Superusuários
plt.subplot(1, 2, 2)
ranking_ciap = df_super['Capitulo_CIAP'].value_counts().head(8)
sns.barplot(x=ranking_ciap.values, y=ranking_ciap.index, palette='viridis')
plt.title(f"Principais Demandas dos Superusuários\n(N={len(super_ids)} pacientes)", fontweight='bold')
plt.xlabel("Total de Atendimentos Acumulados")
plt.tight_layout()
plt.show()

# ==============================================================================
# TABELA DE ORIGEM PARA O ARTIGO
# ==============================================================================
tabela_artigo = ranking_ciap.reset_index()
tabela_artigo.columns = ['Capítulo CIAP', 'Total de Consultas']
tabela_artigo['Proporção (%)'] = (tabela_artigo['Total de Consultas'] / len(df_super) * 100).round(1)

print("\n" + "="*50)
print(f"TABELA DE APOIO: PERFIL DOS SUPERUSUÁRIOS")
print("="*50)
print(tabela_artigo.to_string(index=False))
tabela_artigo.to_csv('tabela_superusuarios.csv', index=False)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO E FILTRAGEM (FOCO EM SUPERUSUÁRIOS)
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print("Arquivo não encontrado.")
    raise

# Identificando Colunas
col_id = 'Record ID'
col_ciap = next((c for c in df.columns if 'ciap' in c.lower() or 'diagnostico' in c.lower()), None)
col_status = next((c for c in df.columns if any(p in c.lower() for p in ['desfecho', 'status'])), None)

# Filtro: Apenas Atendimentos Realizados (Exclui faltas)
df_atend = df[~df[col_status].astype(str).str.contains('não compareceu|falta', case=False, na=False)].copy()

# ==============================================================================
# 2. DEFINIÇÃO ESTATÍSTICA (P95)
# ==============================================================================
contagem = df_atend.groupby(col_id).size().reset_index(name='Total')
limiar_p95 = contagem['Total'].quantile(0.95)

# Identificando quem são e filtrando a base
super_ids = contagem[contagem['Total'] >= limiar_p95][col_id]
df_super = df_atend[df_atend[col_id].isin(super_ids)].copy()

# Mapeamento CIAP-2
ciap_nomes = {
    'A': 'Geral/Inespecífico', 'D': 'Digestivo', 'K': 'Circulatório',
    'L': 'Musculoesquelético', 'P': 'Psicológico', 'R': 'Respiratório',
    'S': 'Pele', 'T': 'Endócrino/Metabólico', 'B': 'Sangue'
}

df_super['CIAP_Label'] = df_super[col_ciap].apply(
    lambda x: ciap_nomes.get(str(x)[0].upper(), 'Outros Diagnósticos')
)

# ==============================================================================
# 3. GERAÇÃO DA BASE DO DIAGRAMA (TABELA ORIGEM-DESTINO)
# ==============================================================================
dist_ciap = df_super['CIAP_Label'].value_counts()

# Criando a tabela de links para o Sankey
n_pacientes = len(super_ids)
n_consultas = len(df_super)

base_links = []
# Link 1: Grupo de Pacientes -> Volume de Consultas
base_links.append(['Superusuários (P95)', 'Total de Consultas', n_consultas])

# Link 2: Volume de Consultas -> Especialidades CIAP
for ciap, valor in dist_ciap.items():
    base_links.append(['Total de Consultas', ciap, valor])

df_base_sankey = pd.DataFrame(base_links, columns=['Source', 'Target', 'Value'])

# ==============================================================================
# 4. RENDERIZAÇÃO DO DIAGRAMA DE SANKEY
# ==============================================================================
# Mapeando labels únicos para índices numéricos
all_nodes = pd.unique(df_base_sankey[['Source', 'Target']].values.ravel('K'))
mapping = {node: i for i, node in enumerate(all_nodes)}

df_base_sankey['source_idx'] = df_base_sankey['Source'].map(mapping)
df_base_sankey['target_idx'] = df_base_sankey['Target'].map(mapping)

# Adicionando N e % nos labels para o gráfico
display_labels = []
for node in all_nodes:
    if node == 'Superusuários (P95)':
        display_labels.append(f"SUPERUSUÁRIOS<br>(N={n_pacientes} Indivíduos)")
    elif node == 'Total de Consultas':
        display_labels.append(f"CARGA DE CONSULTAS<br>(N={n_consultas} Atendimentos)")
    else:
        v = dist_ciap.get(node, 0)
        p = (v/n_consultas*100)
        display_labels.append(f"{node}<br>({v} consultas | {p:.1f}%)")

fig = go.Figure(data=[go.Sankey(
    node = dict(pad=20, thickness=20, line=dict(color="black", width=0.5),
                label=display_labels, color="#e67e22"),
    link = dict(source=df_base_sankey['source_idx'], 
                target=df_base_sankey['target_idx'], 
                value=df_base_sankey['Value'],
                color="rgba(230, 126, 34, 0.3)")
)])

fig.update_layout(title_text=f"Fluxo de Hiperutilização: Distribuição Clínica dos Superusuários (Top 5%)", font_size=11)
fig.show()

# ==============================================================================
# 5. EXIBIÇÃO DA TABELA DE BASE
# ==============================================================================
print("\n" + "="*60)
print("BASE DO DIAGRAMA DE SANKEY (ORIGEM -> DESTINO)")
print("="*60)
print(df_base_sankey[['Source', 'Target', 'Value']].to_string(index=False))
print("-" * 60)
print(f"Limiar de Hiperutilização (P95): >= {limiar_p95:.0f} consultas por paciente.")
df_base_sankey.to_csv('base_sankey_hiperutilizacao.csv', index=False)